# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

- Nama:SHAH FIRIZKI AZMI
- Email:2200018178@webmail.uad.ac.id
- Id Dicoding:shah-firizki-azmi

## Persiapan

### Menyiapkan library yang dibutuhkan

In [1]:
# ============================================================================
# SECTION 1: SETUP ENVIRONMENT - IMPORT ALL REQUIRED LIBRARIES
# ============================================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization & Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Model Training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Model Persistence & Serialization
import joblib

# Utility
import warnings
warnings.filterwarnings('ignore')

print("All required libraries successfully imported!")
print("=" * 70)

All required libraries successfully imported!


### Menyiapkan data yang akan diguankan

## Data Understanding

In [2]:
# ============================================================================
# SECTION 2: LOAD DATA, EXPLORATION, DAN UNDERSTANDING
# VARIABEL YANG DIDEFINISIKAN: df, missing_count, status_counts, status_pct
# ============================================================================

print("\n📊 SECTION 2: MEMUAT DAN MEMAHAMI DATASET")
print("=" * 70)

import os
import urllib.request

# Coba berbagai path possibilities
possible_paths = [
    '../../Dataset/data.csv',
    '../Dataset/data.csv',
    'Dataset/data.csv',
    os.path.join('..', '..', 'Dataset', 'data.csv')
]

df = None  # ✅ DEFINE: df variable

# Coba load dari local path
for path in possible_paths:
    try:
        if os.path.exists(path):
            df = pd.read_csv(path, sep=';')
            print(f"✅ Dataset berhasil dimuat dari: {path}")
            break
    except Exception as e:
        continue

# Jika local path tidak ada, download dari GitHub
if df is None:
    print("📥 Mengunduh dataset dari GitHub...")
    try:
        url = "https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset/main/students_performance/data.csv"
        df = pd.read_csv(url, sep=';')
        print(f"✅ Dataset berhasil diunduh dari GitHub!")
    except Exception as e:
        raise FileNotFoundError(f"Tidak dapat memuat dataset: {e}")

print(f"\n✅ Dataset berhasil dimuat!")
print(f"\n📌 DIMENSI DATASET:")
print(f"   • Total Baris (Records): {df.shape[0]:,}")
print(f"   • Total Kolom (Features): {df.shape[1]}")
print(f"   • Memory Usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

print(f"\n📌 INFORMASI DATASET:")
print(f"   • First Column: {df.columns[0]}")
print(f"   • Last Column: {df.columns[-1]}")

print(f"\n📌 MISSING VALUES:")
missing_count = df.isnull().sum().sum()  # ✅ DEFINE: missing_count
print(f"   • Total Missing Values: {missing_count}")

print(f"\n📌 TARGET VARIABLE (Status) - DISTRIBUSI AWAL:")
status_counts = df['Status'].value_counts()  # ✅ DEFINE: status_counts
status_pct = df['Status'].value_counts(normalize=True) * 100  # ✅ DEFINE: status_pct
for status in status_counts.index:
    print(f"   • {status}: {status_counts[status]:,} records ({status_pct[status]:.1f}%)")

print(f"\n✅ Data Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n✅ Dataset Understanding Complete!")


📊 SECTION 2: MEMUAT DAN MEMAHAMI DATASET
✅ Dataset berhasil dimuat dari: ../Dataset/data.csv

✅ Dataset berhasil dimuat!

📌 DIMENSI DATASET:
   • Total Baris (Records): 4,424
   • Total Kolom (Features): 31
   • Memory Usage: 5.29 MB

📌 INFORMASI DATASET:
   • First Column: Status
   • Last Column: Probability_Graduate

📌 MISSING VALUES:
   • Total Missing Values: 0

📌 TARGET VARIABLE (Status) - DISTRIBUSI AWAL:
   • Graduate: 2,209 records (49.9%)
   • Dropout: 1,421 records (32.1%)
   • Enrolled: 794 records (17.9%)

✅ Data Shape: 4424 rows × 31 columns

✅ Dataset Understanding Complete!


## Data Preparation / Preprocessing

In [3]:
# ============================================================================
# SECTION 3: DATA PREPARATION & PREPROCESSING
# MELIPUTI: Filtering, Separasi Features/Target, Encoding, Scaling, Train-Test Split
# VARIABEL PREREQUISITE: df (dari Section 2)
# VARIABEL YANG DIDEFINISIKAN: df_training, X, y, X_filled, categorical_features,
#                               label_encoders, le_target, y_encoded, X_train, X_test,
#                               y_train, y_test, scaler, X_train_scaled, X_test_scaled
# ============================================================================

print("\n" + "="*70)
print("SECTION 3: DATA PREPARATION & PREPROCESSING (LENGKAP)")
print("="*70)

# ===== STEP 1: DATA FILTERING =====
print("\n[STEP 1] Data Filtering - Hanya Dropout & Graduate untuk Training")
print("-" * 70)

print("""
⚠️ CATATAN PENTING - Mengapa Status "Enrolled" dikecualikan:
   • Dropout & Graduate: Sudah memiliki LABEL AKHIR yang jelas
   • Enrolled: Masih AKTIF belajar - tidak ada label akhir yang definitif
   • Model hanya boleh dilatih dengan data yang memiliki label akhir yang jelas
   • Enrolled hanya digunakan untuk PREDIKSI/INFERENSI di tahap deployment
""")

print("\n📊 STATUS DISTRIBUTION SEBELUM FILTERING:")
status_before = df['Status'].value_counts()
for status in status_before.index:
    print(f"   • {status}: {status_before[status]:,}")
print(f"   Total: {len(df):,} records")

# Filter data untuk hanya Dropout dan Graduate
df_training = df[df['Status'].isin(['Dropout', 'Graduate'])].copy()

print("\n📊 STATUS DISTRIBUTION SESUDAH FILTERING:")
status_after = df_training['Status'].value_counts()
for status in status_after.index:
    pct = (status_after[status] / len(df_training)) * 100
    print(f"   • {status}: {status_after[status]:,} ({pct:.1f}%)")
print(f"   Total: {len(df_training):,} records")
print(f"   ✅ Excluded (Enrolled): {len(df) - len(df_training):,} records (untuk prediksi)")

# ===== STEP 2: SEPARASI FEATURES & TARGET =====
print("\n[STEP 2] Separasi Features (X) dan Target (y)")
print("-" * 70)

X = df_training.drop('Status', axis=1)
y = df_training['Status']

print(f"✅ Features (X): {X.shape[0]} rows × {X.shape[1]} columns")
print(f"✅ Target (y): {y.shape[0]} rows")
print(f"\n   First 5 columns: {X.columns[:5].tolist()}")
print(f"   Last 5 columns: {X.columns[-5:].tolist()}")

# ===== STEP 3: ENCODING CATEGORICAL VARIABLES =====
print("\n[STEP 3] Encode Categorical Variables")
print("-" * 70)

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

print(f"Categorical features found: {len(categorical_features)}")
for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"   ✅ {col}: {len(le.classes_)} unique values")

# ===== STEP 4: ENCODE TARGET VARIABLE =====
print("\n[STEP 4] Encode Target Variable (Status - ONLY 2 CLASSES)")
print("-" * 70)

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print("Target variable encoding:")
for idx, label in enumerate(le_target.classes_):
    count = (y_encoded == idx).sum()
    pct = (count / len(y_encoded)) * 100
    print(f"   • {label} → {idx} ({count:,} records, {pct:.1f}%)")

# ===== STEP 5: HANDLE MISSING VALUES =====
print("\n[STEP 5] Handle Missing Values")
print("-" * 70)

missing_before = X.isnull().sum().sum()
X_filled = X.fillna(X.mean(numeric_only=True))
missing_after = X_filled.isnull().sum().sum()

print(f"Missing values before: {missing_before}")
print(f"Missing values after: {missing_after} ✅")

# ===== STEP 6: TRAIN-TEST SPLIT =====
print("\n[STEP 6] Train-Test Split (80-20 dengan Stratification)")
print("-" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"✅ Training Set: {X_train.shape[0]:,} records ({(X_train.shape[0]/len(X_filled)*100):.1f}%)")
print(f"   • {(y_train == 0).sum():,} class 0 (Dropout)")
print(f"   • {(y_train == 1).sum():,} class 1 (Graduate)")
print(f"\n✅ Test Set: {X_test.shape[0]:,} records ({(X_test.shape[0]/len(X_filled)*100):.1f}%)")
print(f"   • {(y_test == 0).sum():,} class 0 (Dropout)")
print(f"   • {(y_test == 1).sum():,} class 1 (Graduate)")

# Verify stratification
print(f"\n✅ Stratification Check:")
train_ratio = (y_train == 0).sum() / len(y_train) * 100
test_ratio = (y_test == 0).sum() / len(y_test) * 100
overall_ratio = (y_encoded == 0).sum() / len(y_encoded) * 100
print(f"   • Overall class 0 ratio: {overall_ratio:.1f}%")
print(f"   • Train class 0 ratio: {train_ratio:.1f}%")
print(f"   • Test class 0 ratio: {test_ratio:.1f}%")

# ===== STEP 7: FEATURE SCALING =====
print("\n[STEP 7] Feature Scaling - StandardScaler")
print("-" * 70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ StandardScaler fitted on training data")
print(f"   • Train data - Mean (≈0): {X_train_scaled.mean(axis=0)[:5]}")
print(f"   • Train data - Std (≈1): {X_train_scaled.std(axis=0)[:5]}")
print(f"   ✅ Scaling applied to both training and test sets")

print("\n" + "="*70)
print("✅ DATA PREPARATION & PREPROCESSING - COMPLETE")
print(f"   Data ready for model training: {X_train_scaled.shape[0]} training samples")
print("="*70)


SECTION 3: DATA PREPARATION & PREPROCESSING (LENGKAP)

[STEP 1] Data Filtering - Hanya Dropout & Graduate untuk Training
----------------------------------------------------------------------

⚠️ CATATAN PENTING - Mengapa Status "Enrolled" dikecualikan:
   • Dropout & Graduate: Sudah memiliki LABEL AKHIR yang jelas
   • Enrolled: Masih AKTIF belajar - tidak ada label akhir yang definitif
   • Model hanya boleh dilatih dengan data yang memiliki label akhir yang jelas
   • Enrolled hanya digunakan untuk PREDIKSI/INFERENSI di tahap deployment


📊 STATUS DISTRIBUTION SEBELUM FILTERING:
   • Graduate: 2,209
   • Dropout: 1,421
   • Enrolled: 794
   Total: 4,424 records

📊 STATUS DISTRIBUTION SESUDAH FILTERING:
   • Graduate: 2,209 (60.9%)
   • Dropout: 1,421 (39.1%)
   Total: 3,630 records
   ✅ Excluded (Enrolled): 794 records (untuk prediksi)

[STEP 2] Separasi Features (X) dan Target (y)
----------------------------------------------------------------------
✅ Features (X): 3630 rows × 30

## Modeling

In [4]:
# ============================================================================
# SECTION 4: MODEL TRAINING - TRAIN 3 ALGORITHMS & COMPARE PERFORMANCE
# VARIABEL PREREQUISITE: X_train_scaled, X_test_scaled, y_train, y_test, le_target
# VARIABEL YANG DIDEFINISIKAN: rf_model, gb_model, lr_model, y_pred_rf, y_pred_gb, 
#                               y_pred_lr, best_model, best_model_name
# ============================================================================

print("\n" + "="*70)
print("SECTION 4: MODEL TRAINING (3 ALGORITMA)")
print("="*70)

# ===== MODEL 1: RANDOM FOREST =====
print("\n[MODEL 1] Random Forest Classifier")
print("-" * 70)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Hyperparameters:")
print(f"   • n_estimators: 100")
print(f"   • max_depth: 15")
print(f"   • min_samples_split: 5")
print(f"   • min_samples_leaf: 2")
print(f"   • random_state: 42")

rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print(f"\n✅ Model trained successfully!")
print(f"   • Training set accuracy: {rf_model.score(X_train_scaled, y_train):.4f}")
print(f"   • Test set accuracy: {rf_accuracy:.4f}")

# ===== MODEL 2: GRADIENT BOOSTING =====
print("\n[MODEL 2] Gradient Boosting Classifier")
print("-" * 70)

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

print("Hyperparameters:")
print(f"   • n_estimators: 100")
print(f"   • learning_rate: 0.1")
print(f"   • max_depth: 5")
print(f"   • min_samples_split: 5")
print(f"   • min_samples_leaf: 2")
print(f"   • random_state: 42")

gb_model.fit(X_train_scaled, y_train)
y_pred_gb = gb_model.predict(X_test_scaled)
gb_accuracy = accuracy_score(y_test, y_pred_gb)

print(f"\n✅ Model trained successfully!")
print(f"   • Training set accuracy: {gb_model.score(X_train_scaled, y_train):.4f}")
print(f"   • Test set accuracy: {gb_accuracy:.4f}")

# ===== MODEL 3: LOGISTIC REGRESSION =====
print("\n[MODEL 3] Logistic Regression")
print("-" * 70)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

print("Hyperparameters:")
print(f"   • max_iter: 1000")
print(f"   • random_state: 42")

lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, y_pred_lr)

print(f"\n✅ Model trained successfully!")
print(f"   • Training set accuracy: {lr_model.score(X_train_scaled, y_train):.4f}")
print(f"   • Test set accuracy: {lr_accuracy:.4f}")

# ===== MODEL COMPARISON =====
print("\n" + "-" * 70)
print("📊 MODEL COMPARISON (Test Set Accuracy)")
print("-" * 70)

models_comparison = {
    'Random Forest': rf_accuracy,
    'Gradient Boosting': gb_accuracy,
    'Logistic Regression': lr_accuracy
}

for model_name, accuracy in sorted(models_comparison.items(), key=lambda x: x[1], reverse=True):
    rank = "🥇 BEST" if accuracy == max(models_comparison.values()) else ""
    print(f"   • {model_name}: {accuracy:.4f} {rank}")

# Select best model
best_accuracy = max(models_comparison.values())
best_model_name = [k for k, v in models_comparison.items() if v == best_accuracy][0]
best_model = {'Random Forest': rf_model, 'Gradient Boosting': gb_model, 'Logistic Regression': lr_model}[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name} (Accuracy: {best_accuracy:.4f})")

print("\n" + "="*70)
print("✅ MODEL TRAINING - COMPLETE")
print("="*70)


SECTION 4: MODEL TRAINING (3 ALGORITMA)

[MODEL 1] Random Forest Classifier
----------------------------------------------------------------------
Hyperparameters:
   • n_estimators: 100
   • max_depth: 15
   • min_samples_split: 5
   • min_samples_leaf: 2
   • random_state: 42

✅ Model trained successfully!
   • Training set accuracy: 0.9997
   • Test set accuracy: 0.9780

[MODEL 2] Gradient Boosting Classifier
----------------------------------------------------------------------
Hyperparameters:
   • n_estimators: 100
   • learning_rate: 0.1
   • max_depth: 5
   • min_samples_split: 5
   • min_samples_leaf: 2
   • random_state: 42

✅ Model trained successfully!
   • Training set accuracy: 1.0000
   • Test set accuracy: 0.9766

[MODEL 3] Logistic Regression
----------------------------------------------------------------------
Hyperparameters:
   • max_iter: 1000
   • random_state: 42

✅ Model trained successfully!
   • Training set accuracy: 0.9301
   • Test set accuracy: 0.9242

-

## Evaluation

In [5]:
# ============================================================================
# SECTION 5: MODEL EVALUATION - METRICS, CONFUSION MATRIX, MODEL SAVING
# VARIABEL PREREQUISITE: best_model, y_test, y_pred_rf, y_pred_gb, y_pred_lr
# VARIABEL YANG DIDEFINISIKAN: classification_reports (dict)
# ============================================================================

print("\n" + "="*70)
print("SECTION 5: MODEL EVALUATION - PERFORMA & METRICS")
print("="*70)

# ===== EVALUATION FUNCTION =====
def evaluate_model(model_name, y_true, y_pred, y_prob=None):
    print(f"\n📊 {model_name} - DETAILED METRICS")
    print("-" * 70)
    
    # Accuracy
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    
    # F1 Score
    f1 = f1_score(y_true, y_pred)
    print(f"F1 Score: {f1:.4f}")
    
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, 
                               target_names=['Dropout', 'Graduate'],
                               digits=4))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:")
    print(f"   Expected Dropout, Predicted Dropout: {cm[0,0]}")
    print(f"   Expected Dropout, Predicted Graduate: {cm[0,1]}")
    print(f"   Expected Graduate, Predicted Dropout: {cm[1,0]}")
    print(f"   Expected Graduate, Predicted Graduate: {cm[1,1]}")
    
    return acc, f1, cm

# ===== EVALUATE ALL MODELS =====
print("\n[Random Forest]\n")
rf_acc, rf_f1, rf_cm = evaluate_model('Random Forest', y_test, y_pred_rf)

print("\n[Gradient Boosting]\n")
gb_acc, gb_f1, gb_cm = evaluate_model('Gradient Boosting', y_test, y_pred_gb)

print("\n[Logistic Regression]\n")
lr_acc, lr_f1, lr_cm = evaluate_model('Logistic Regression', y_test, y_pred_lr)

# ===== SUMMARY COMPARISON =====
print("\n" + "=" * 70)
print("📊 SUMMARY: PERBANDINGAN SEMUA MODEL")
print("=" * 70)

comparison_data = {
    'Model': ['Random Forest', 'Gradient Boosting', 'Logistic Regression'],
    'Accuracy': [rf_acc, gb_acc, lr_acc],
    'F1 Score': [rf_f1, gb_f1, lr_f1]
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n" + df_comparison.to_string(index=False))

best_idx = df_comparison['Accuracy'].idxmax()
best_model_final = df_comparison.loc[best_idx, 'Model']
best_accuracy_final = df_comparison.loc[best_idx, 'Accuracy']

print(f"\n🏆 BEST PERFORMING MODEL: {best_model_final}")
print(f"   Accuracy: {best_accuracy_final:.4f} ({best_accuracy_final*100:.2f}%)")

# ===== SAVE MODEL & ARTIFACTS =====
print("\n" + "=" * 70)
print("💾 MENYIMPAN MODEL & PREPROCESSING ARTIFACTS")
print("=" * 70)

import os

# Create model directory if not exists
model_dir = 'model'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"✅ Created directory: {model_dir}")

# Save the best model (Random Forest)
model_path = os.path.join(model_dir, 'student_status_model.joblib')
joblib.dump(rf_model, model_path)
print(f"✅ Model saved: {model_path}")

# Save scaler
scaler_path = os.path.join(model_dir, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler saved: {scaler_path}")

# Save label encoders
encoders_path = os.path.join(model_dir, 'label_encoders.joblib')
joblib.dump(label_encoders, encoders_path)
print(f"✅ Label Encoders saved: {encoders_path}")

# Save target encoder
target_encoder_path = os.path.join(model_dir, 'le_target.joblib')
joblib.dump(le_target, target_encoder_path)
print(f"✅ Target Encoder saved: {target_encoder_path}")

# Save feature names
feature_names_path = os.path.join(model_dir, 'feature_names.joblib')
joblib.dump(X_train.columns.tolist(), feature_names_path)
print(f"✅ Feature Names saved: {feature_names_path}")

print("\n" + "=" * 70)
print("✅ MODEL EVALUATION & SAVING - COMPLETE")
print("=" * 70)
print("\n🎉 PIPELINE DATA SCIENCE SELESAI!")
print("\nModel yang tersimpan siap untuk deployment/production use.")
print("=" * 70)


SECTION 5: MODEL EVALUATION - PERFORMA & METRICS

[Random Forest]


📊 Random Forest - DETAILED METRICS
----------------------------------------------------------------------
Accuracy: 0.9780 (97.80%)
F1 Score: 0.9819

Classification Report:
              precision    recall  f1-score   support

     Dropout     0.9685    0.9754    0.9719       284
    Graduate     0.9841    0.9796    0.9819       442

    accuracy                         0.9780       726
   macro avg     0.9763    0.9775    0.9769       726
weighted avg     0.9780    0.9780    0.9780       726

Confusion Matrix:
   Expected Dropout, Predicted Dropout: 277
   Expected Dropout, Predicted Graduate: 7
   Expected Graduate, Predicted Dropout: 9
   Expected Graduate, Predicted Graduate: 433

[Gradient Boosting]


📊 Gradient Boosting - DETAILED METRICS
----------------------------------------------------------------------
Accuracy: 0.9766 (97.66%)
F1 Score: 0.9808

Classification Report:
              precision    recall  f1